# 07 - Comparacao dos modelos finais

Compara os modelos finais registrados no MLflow para Random Forest Regressor e XGBoost Regressor. A tabela inclui metricas de erro, ganho contra baseline e tamanho do artefato do modelo.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR is None:
    raise RuntimeError("Nao encontrei ml/notebooks/utils. Execute o notebook dentro do repositorio Saltim.")
sys.path.insert(0, str(UTILS_DIR))

import pandas as pd

from regressor_tuning_common import (
    compare_final_model_runs,
    log_final_model_comparison,
    plot_final_model_comparison,
)


## Comparacao registrada no MLflow

Execute os notebooks `05` e `06` antes deste notebook. A comparacao usa os runs champion mais recentes de cada experimento final.

In [ ]:
comparison = compare_final_model_runs()

if comparison.empty:
    display(pd.DataFrame({"aviso": ["Nenhum run champion encontrado. Execute os notebooks 05 e 06 primeiro."]}))
else:
    display(
        comparison[[
            "model",
            "dataset_mode",
            "test_rmse",
            "test_mae",
            "test_r2",
            "best_cv_rmse",
            "baseline_test_rmse",
            "test_rmse_gain_vs_baseline",
            "model_size_mb",
            "registered_model_name",
            "run_id",
        ]]
    )


In [ ]:
comparison_plot = plot_final_model_comparison(comparison)
comparison_plot


In [ ]:
comparison_mlflow_info = log_final_model_comparison(comparison, comparison_plot)
comparison_mlflow_info


## Ranking

O ranking principal usa menor `test_rmse`. Em caso de desempenho parecido, `model_size_mb` ajuda a escolher um modelo mais leve para servir em producao.

In [ ]:
if not comparison.empty:
    ranking = comparison.sort_values(["test_rmse", "model_size_mb"], ascending=[True, True]).reset_index(drop=True)
    ranking.insert(0, "rank", ranking.index + 1)
    display(ranking[["rank", "model", "test_rmse", "model_size_mb", "model_uri"]])
